# Aula 6 — Séries Temporais com Prophet e TimesFM

Na aula passada, a régua (copiar a semana anterior) errava R\\$ 63,78 por
dia, e a regressão com calendário, montada à mão, errava R\\$ 44,11. Hoje
vamos testar dois atalhos: o **Prophet**, que monta a regressão com
calendário por você, e o **TimesFM**, um modelo pré-treinado que prevê a
série sem treinar nada.

**Como usar este notebook:** rode as células de cima para baixo,
acompanhando a explicação. Cada célula faz uma coisa só, e o texto antes
dela diz o quê.

## 0. As bibliotecas

Esta célula confere se o Prophet e uma versão recente do `transformers`
(a biblioteca que traz o TimesFM) estão instalados, e instala o que
faltar. Rode ela primeiro e espere terminar: pode levar um minuto.

In [ ]:
!pip install -q prophet "transformers>=5.3"

O resto já vem no Colab:

- `pandas`, `numpy` e `matplotlib`, as de sempre;
- `scikit-learn` para as métricas de erro;
- `prophet` para o Prophet;
- `torch` (PyTorch) e `transformers` para o TimesFM.

In [ ]:
import logging
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error

from prophet import Prophet

import torch
from transformers import TimesFm2_5ModelForPrediction

# Esconde avisos técnicos e as mensagens de progresso do Prophet
warnings.filterwarnings("ignore")
logging.getLogger("cmdstanpy").disabled = True

## 1. Os dados e o formato que o Prophet exige

O Prophet só aceita duas colunas, com estes nomes exatos:

| Coluna | O que é |
|---|---|
| `ds` | a data |
| `y` | o valor que você quer prever |

Por isso o primeiro passo é renomear as colunas com o `.rename`. A
coluna `feriado` fica guardada à parte, porque vamos precisar dela
depois.

In [ ]:
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/vendas_cafeteria.csv"

dados = pd.read_csv(URL_DADOS, parse_dates=["data"])
dados = dados.rename(columns={"data": "ds", "vendas": "y"})
dados.head()

O mesmo corte no tempo da aula passada: os últimos 28 dias ficam
escondidos. O treino que vai para o Prophet leva só as colunas `ds` e
`y`.

In [ ]:
treino = dados.iloc[:-28]
teste = dados.iloc[-28:].copy()

treino_prophet = treino[["ds", "y"]]

print("Treino:", len(treino), "dias, até", treino["ds"].max().date())
print("Teste: ", len(teste), "dias, a partir de", teste["ds"].min().date())

A régua da Aula 5: repetir a última semana do treino 4 vezes. Assim
como na aula passada, cada previsão vira uma coluna da tabela `teste`.

In [ ]:
ultima_semana = treino["y"].iloc[-7:]
teste["regua"] = np.tile(ultima_semana, 4)

mae_regua = mean_absolute_error(teste["y"], teste["regua"])
print(f"A régua: MAE R$ {mae_regua:.2f}")

## 2. O Prophet

O modelo do Prophet é a soma das peças que você já conhece:

$$y(t) = g(t) + s(t) + h(t) + \varepsilon_t$$

Onde $g(t)$ é a tendência, $s(t)$ são as sazonalidades, $h(t)$ são os
feriados e $\varepsilon_t$ é o que sobra. Treinar e prever são três
passos:

1. `Prophet()` cria o modelo e `.fit(...)` treina;
2. `.make_future_dataframe(periods=28)` monta a tabela de datas: o
   histórico inteiro mais os 28 dias novos;
3. `.predict(...)` preenche a previsão de cada data.

In [ ]:
modelo = Prophet()
modelo.fit(treino_prophet)

futuro = modelo.make_future_dataframe(periods=28)
previsao = modelo.predict(futuro)

previsao[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()

Três colunas importam: `yhat` é a previsão, e `yhat_lower` e
`yhat_upper` são as bordas da faixa de incerteza. O Prophet tem um
gráfico pronto para isso:

In [ ]:
modelo.plot(previsao)
plt.title("Vendas observadas e previsão do Prophet")
plt.xlabel("Data")
plt.ylabel("Vendas (R$)")
plt.show()

### O Prophet bateu a régua?

A `previsao` cobre o histórico inteiro **mais** os 28 dias novos. Para
comparar com o teste, pegamos só as últimas 28 linhas do `yhat`.

In [ ]:
teste["prophet"] = previsao["yhat"].iloc[-28:].to_numpy()

mae_prophet = mean_absolute_error(teste["y"], teste["prophet"])
print(f"Prophet: MAE R$ {mae_prophet:.2f}")
print(f"A régua: MAE R$ {mae_regua:.2f}")

Empate. A biblioteca famosa, direto da caixa, erra o mesmo que copiar a
semana passada.

### As peças que o modelo estimou sozinho

Este é o gráfico que vale a reunião: cada peça do modelo, separada.

In [ ]:
modelo.plot_components(previsao)
plt.show()

## 3. O que o modelo não adivinha: o calendário

O Prophet descobre tendência e sazonalidade sozinho. Feriado, não: ele
não tem como saber que 25 de dezembro é diferente. A lista de datas
especiais é conhecimento de negócio, e quem entrega é você.

O Prophet pede a lista numa tabela com duas colunas: `holiday` (um nome
para o grupo) e `ds` (as datas).

In [ ]:
datas_de_feriado = dados[dados["feriado"] == 1]["ds"]

feriados = pd.DataFrame({
    "holiday": "feriado_nacional",
    "ds": datas_de_feriado,
})
feriados.tail()

O modelo novo é igual ao anterior, com um argumento a mais:
`holidays=feriados`.

In [ ]:
modelo_feriados = Prophet(holidays=feriados)
modelo_feriados.fit(treino_prophet)

futuro_feriados = modelo_feriados.make_future_dataframe(periods=28)
previsao_feriados = modelo_feriados.predict(futuro_feriados)

teste["prophet_feriados"] = previsao_feriados["yhat"].iloc[-28:].to_numpy()

mae_prophet_feriados = mean_absolute_error(teste["y"], teste["prophet_feriados"])
print(f"Prophet com feriados:              MAE R$ {mae_prophet_feriados:.2f}")
print(f"Regressão com calendário (Aula 5): MAE R$ 44.11")

Com a lista de feriados, o Prophet chega praticamente ao mesmo número da
regressão da Aula 5. Não é coincidência: os dois somam as mesmas peças.

### O intervalo de incerteza

Cada dia previsto vem com `yhat_lower` e `yhat_upper`, as bordas de uma
faixa que promete cobrir 80% dos casos. Quantos dias do teste caíram
dentro dela?

In [ ]:
teste["prophet_baixo"] = previsao_feriados["yhat_lower"].iloc[-28:].to_numpy()
teste["prophet_cima"] = previsao_feriados["yhat_upper"].iloc[-28:].to_numpy()

dentro_da_faixa = (teste["y"] >= teste["prophet_baixo"]) & (teste["y"] <= teste["prophet_cima"])
largura = teste["prophet_cima"] - teste["prophet_baixo"]

print(f"Dias dentro da faixa: {dentro_da_faixa.sum()} de 28 ({dentro_da_faixa.mean():.0%})")
print(f"Largura média da faixa: R$ {largura.mean():.2f}")

## 4. Um modelo que nunca viu esta cafeteria: o TimesFM

O **TimesFM**, do Google, é um **modelo de fundação** para séries
temporais. Ele leu uma quantidade enorme de séries (buscas no Google,
visitas à Wikipédia e muitas outras) e aprendeu a prever o próximo
pedaço de uma série, como um modelo de linguagem prevê a próxima
palavra.

Por isso ele prevê uma série nova **sem treinar nela**. Não existe
`fit`: você entrega o passado, e ele devolve o futuro.

O `from_pretrained` baixa o modelo pronto do Hugging Face (cerca de 1 GB,
só na primeira vez). A primeira linha escolhe onde ele vai rodar: na
GPU, se o Colab tiver uma, ou no processador, que também dá conta em
poucos segundos.

In [ ]:
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Rodando em:", dispositivo)

timesfm = TimesFm2_5ModelForPrediction.from_pretrained("google/timesfm-2.5-200m-transformers")
timesfm = timesfm.to(dispositivo)

total_de_pesos = timesfm.num_parameters()
print(f"O TimesFM tem {total_de_pesos / 1_000_000:.0f} milhões de pesos, e nenhum deles vai mudar hoje.")

### A previsão

O modelo recebe o passado como **tensor**, o nome que o PyTorch dá para
uma lista de números. O `torch.no_grad()` avisa que só vamos prever,
sem treinar, o que deixa a conta mais rápida.

A saída traz duas coisas:

- `mean_predictions`: a previsão, dia a dia;
- `full_predictions`: os **quantis**, as bordas de uma faixa. A coluna
  1 é o quantil de 10% e a coluna 9 é o de 90%: entre elas devem cair 80%
  dos dias, a mesma promessa do Prophet.

O modelo sempre prevê 128 dias à frente, por isso o `[0, :28]`: a
primeira (e única) série, os primeiros 28 dias.

In [ ]:
historico = torch.tensor(treino["y"].to_numpy(), dtype=torch.float32, device=dispositivo)

with torch.no_grad():
    saida = timesfm(past_values=[historico])

teste["timesfm"] = saida.mean_predictions[0, :28].cpu().numpy()
teste["timesfm_baixo"] = saida.full_predictions[0, :28, 1].cpu().numpy()
teste["timesfm_cima"] = saida.full_predictions[0, :28, 9].cpu().numpy()

mae_timesfm = mean_absolute_error(teste["y"], teste["timesfm"])
print(f"TimesFM, sem treino: MAE R$ {mae_timesfm:.2f}")
print(f"A régua:             MAE R$ {mae_regua:.2f}")

In [ ]:
plt.figure(figsize=(11, 4))
plt.fill_between(teste["ds"], teste["timesfm_baixo"], teste["timesfm_cima"], alpha=0.2, label="Faixa de 10% a 90%")
plt.plot(teste["ds"], teste["y"], label="O que aconteceu", linewidth=3)
plt.plot(teste["ds"], teste["timesfm"], label="TimesFM")
plt.xlabel("Data")
plt.ylabel("Vendas (R$)")
plt.legend()
plt.show()

Sem treinar nada, ele bate a régua e bate o Prophet sem feriados. Mas
repare no dia 25. Vamos separar o Natal dos outros dias.

In [ ]:
natal = teste[teste["feriado"] == 1]
dias_comuns = teste[teste["feriado"] == 0]

print(f"No Natal: previu R$ {natal['timesfm'].iloc[0]:.2f}, vendeu R$ {natal['y'].iloc[0]:.2f}")

mae_dias_comuns = mean_absolute_error(dias_comuns["y"], dias_comuns["timesfm"])
print(f"Erro médio nos outros 27 dias: R$ {mae_dias_comuns:.2f}")

Nos dias comuns, ele é o melhor de todos. O problema é um dia só: o
TimesFM não tem como saber que 25 de dezembro é feriado.

### Quanto passado entregar?

O TimesFM não tem pesos para treinar, mas você escolhe o **contexto**:
quantos dias de passado ele recebe. Acima entregamos todos os 1.068
dias. Com só os últimos 365, o `.iloc[-365:]`:

In [ ]:
historico_365 = torch.tensor(treino["y"].iloc[-365:].to_numpy(), dtype=torch.float32, device=dispositivo)

with torch.no_grad():
    saida_365 = timesfm(past_values=[historico_365])

teste["timesfm_365"] = saida_365.mean_predictions[0, :28].cpu().numpy()

mae_365 = mean_absolute_error(teste["y"], teste["timesfm_365"])
print(f"TimesFM com 365 dias de contexto:   MAE R$ {mae_365:.2f}")
print(f"TimesFM com 1.068 dias de contexto: MAE R$ {mae_timesfm:.2f}")

Com mais passado, o modelo enxerga também o ciclo do ano, e descobre que
dezembro é baixa temporada. Na dúvida, entregue todo o histórico.

## 5. Juntando os dois mundos

O TimesFM traz o padrão; o calendário é você quem traz. A ideia é somar
à previsão dele o efeito de um feriado, medido no treino:

$$\hat{y}_t = \text{TimesFM}_t + w_1 \cdot \text{feriado}_t$$

O $w_1$ é a venda média de um feriado menos a de um dia comum. Nos dias
comuns, $\text{feriado}_t$ vale 0 e nada muda.

In [ ]:
media_por_feriado = treino.groupby("feriado")["y"].mean()
media_por_feriado.round(2)

In [ ]:
efeito_feriado = media_por_feriado[1] - media_por_feriado[0]
print(f"Um feriado vende R$ {efeito_feriado:.2f} em relação a um dia comum.")

In [ ]:
teste["timesfm_feriado"] = teste["timesfm"] + efeito_feriado * teste["feriado"]

mae_timesfm_feriado = mean_absolute_error(teste["y"], teste["timesfm_feriado"])
print(f"TimesFM + feriado: MAE R$ {mae_timesfm_feriado:.2f}")

O melhor resultado do módulo. O modelo trouxe o padrão, você trouxe o
calendário.

### A faixa do TimesFM

Entre o quantil de 10% e o de 90% devem cair 80% dos dias, a mesma
promessa da faixa do Prophet.

In [ ]:
dentro_timesfm = (teste["y"] >= teste["timesfm_baixo"]) & (teste["y"] <= teste["timesfm_cima"])
largura_timesfm = teste["timesfm_cima"] - teste["timesfm_baixo"]

print(f"TimesFM: {dentro_timesfm.sum()} de 28 dias dentro da faixa ({dentro_timesfm.mean():.0%})")
print(f"Largura média: R$ {largura_timesfm.mean():.2f}")

## 6. O placar do módulo

Todas as previsões estão lado a lado na tabela `teste`. Medimos o erro de
cada coluna e acrescentamos os dois resultados da Aula 5, que entram
prontos.

In [ ]:
nomes = [
    "Sazonal ingênua (a régua)",
    "Prophet, sem feriados",
    "Prophet, com feriados",
    "TimesFM, sem treino",
    "TimesFM + feriado",
]
colunas = ["regua", "prophet", "prophet_feriados", "timesfm", "timesfm_feriado"]

erros = []
for coluna in colunas:
    erros.append(mean_absolute_error(teste["y"], teste[coluna]))

nomes.append("Regressão com calendário (Aula 5)")
erros.append(44.11)
nomes.append("ARIMA sazonal (Aula 5)")
erros.append(57.57)

placar = pd.DataFrame({"Previsão": nomes, "MAE (R$)": erros})
placar = placar.sort_values("MAE (R$)")
placar.round(2)

In [ ]:
plt.figure(figsize=(10, 5))
plt.barh(placar["Previsão"], placar["MAE (R$)"])
plt.axvline(mae_regua, color="black", linestyle="--")
plt.gca().invert_yaxis()
plt.xlabel("MAE no teste (R$)")
plt.title("O placar do módulo (a linha tracejada é a régua)")
plt.show()

Os três melhores vieram de ferramentas bem diferentes: uma regressão,
uma biblioteca de negócio e uma rede neural pré-treinada. O que eles têm
em comum é o calendário de feriados, que nenhum deles descobriu sozinho.